In [2]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
import joblib

# 1. Load data from the first notebook
# Assuming you saved the combined Kaggle data as 'processed_kaggle_sepsis.parquet'
# df = pd.read_csv(r'D:\Uni\Year_3_sem_2\XAI\Project\phase_2\Dataset.csv')
df = pd.read_parquet('processed_kaggle_sepsis.parquet')

In [3]:

# 2. Define Feature Groups based on your CSV snippet
# Dynamic Features: Vitals and Labs that change hourly
dynamic_cols = [
    'HR', 'O2Sat', 'Temp', 'SBP', 'MAP', 'DBP', 'Resp', 'EtCO2',
    'BaseExcess', 'HCO3', 'FiO2', 'pH', 'PaCO2', 'SaO2', 'AST', 'BUN',
    'Alkalinephos', 'Calcium', 'Chloride', 'Creatinine', 'Bilirubin_direct',
    'Glucose', 'Lactate', 'Magnesium', 'Phosphate', 'Potassium',
    'Bilirubin_total', 'Hct', 'Hgb', 'PTT', 'WBC', 'Fibrinogen', 'Platelets'
]

# Static Features: The "Context Vector" described in the paper (Section 3.2.1)
static_cols = ['Age', 'Gender', 'Unit1', 'Unit2', 'HospAdmTime']

In [4]:

# 3. Handle Missing Values (Paper Technique: Forward Fill)
# The paper notes that clinical data is sparse. We forward-fill (carry last measurement forward)
# and fill remaining (start of stay) with 0.
df[dynamic_cols] = df.groupby('Patient_ID')[dynamic_cols].ffill().fillna(0)

# 4. Normalization (Paper Technique: Standard Scaling)
scaler = StandardScaler()
df[dynamic_cols + ['Age', 'HospAdmTime']] = scaler.fit_transform(df[dynamic_cols + ['Age', 'HospAdmTime']])

In [5]:
# 5. Create 3D Temporal Sequences
# The paper's CNN-LSTM requires a sequence for each patient contact.
# We will use a 72-hour lookback window (adjustable).
def create_3d_sequences(group, max_len=72):
    # Extract Dynamic Sequence (Time, Features)
    dyn_seq = group[dynamic_cols].values
    
    # Extract Static Context (1, Features) - only need the first row
    stat_context = group[static_cols].values[0]
    
    # Target Label (per time step)
    labels = group['SepsisLabel'].values
    
    # Padding/Truncating logic to ensure fixed sequence length for the CNN-LSTM
    curr_len = len(dyn_seq)
    if curr_len > max_len:
        dyn_seq = dyn_seq[:max_len]
        labels = labels[:max_len]
    else:
        # Pre-padding with 0s to keep it "causal" (past context)
        pad_size = max_len - curr_len
        dyn_seq = np.pad(dyn_seq, ((pad_size, 0), (0, 0)), mode='constant')
        labels = np.pad(labels, (pad_size, 0), mode='constant')
        
    return dyn_seq, stat_context, labels

# Process each patient
X_dynamic = []
X_static = []
y = []

patient_groups = df.groupby('Patient_ID')
for name, group in patient_groups:
    d, s, l = create_3d_sequences(group)
    X_dynamic.append(d)
    X_static.append(s)
    y.append(l)

# Convert to final Numpy arrays for modeling
X_dynamic = np.array(X_dynamic) # Shape: (Patients, 72, 33)
X_static = np.array(X_static)   # Shape: (Patients, 5)
y = np.array(y)                 # Shape: (Patients, 72)

# 6. Save Artifacts
artifacts = {
    'X_dynamic': X_dynamic,
    'X_static': X_static,
    'y': y,
    'features': dynamic_cols,
    'context': static_cols
}
joblib.dump(artifacts, 'cnn_lstm_data.pkl')

print("Feature Engineering Complete.")
print(f"Sequence Shape: {X_dynamic.shape}")
print(f"Context Shape: {X_static.shape}")

Feature Engineering Complete.
Sequence Shape: (40336, 72, 33)
Context Shape: (40336, 5)
